In [5]:
import pandas as pd
import json
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
import numpy as np

# ---------------------------
# 1. Load data
# ---------------------------
df = pd.read_csv("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/football_test/team_stats.csv")

with open("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/football_test/match_api_metric_map.json") as f:
    mapping = json.load(f)

# Keep only numeric columns
num = df.select_dtypes(include=["float64", "int64"]).copy()

# Remove columns with any missing values (FA cannot handle NaN)
num = num.dropna(axis=1)

# ---------------------------
# 2. Standardize
# ---------------------------
X = StandardScaler().fit_transform(num)

# ---------------------------
# 3. Determine optimal n-factors (1–10)
# ---------------------------
scores = []
for n in range(1, 11):
    fa = FactorAnalysis(n_components=n, random_state=0)
    fa.fit(X)
    scores.append(fa.score(X))

optimal_n = np.argmax(scores) + 1
print("Optimal number of factors =", optimal_n)

# ---------------------------
# 4. Fit final factor model
# ---------------------------
fa = FactorAnalysis(n_components=optimal_n, random_state=0)
fa.fit(X)

loadings = pd.DataFrame(
    fa.components_.T,
    index=num.columns,
    columns=[f"Factor_{i+1}" for i in range(optimal_n)]
)

print("\nFactor Loadings:")
print(loadings)

# ---------------------------
# 5. Auto‑name factors based on strongest loading variables
# ---------------------------
factor_names = {}

for i in range(optimal_n):
    col = f"Factor_{i+1}"

    # find top 5 variables contributing to this factor
    top_vars = (
        loadings[col]
        .abs()
        .sort_values(ascending=False)
        .head(5)
        .index.tolist()
    )

    # Convert technical metric names to human labels from JSON mapping (if exist)
    readable = [mapping.get(v, v) for v in top_vars]

    # Construct a suggested name
    factor_names[col] = " / ".join(readable[:3])

print("\nSuggested Factor Names:")
for k, v in factor_names.items():
    print(f"{k}: {v}")

Optimal number of factors = 10

Factor Loadings:
                                      Factor_1  Factor_2  Factor_3  \
team_id                              -0.022658  0.072397 -0.107351   
competition_id                       -0.034651  0.058539  0.111039   
season                               -0.000000 -0.000000 -0.000000   
fouls_commited                       -0.311888  0.031733  0.164978   
num_throwins_final_third              0.014931 -0.170878  0.214990   
...                                        ...       ...       ...   
opp_shots_from_outside_box_pct        0.129633  0.137647  0.188898   
opp_shots_per_final_third_pass        0.178945 -0.006031 -0.126037   
opp_shots_from_direct_attacks_pct     0.420685 -0.210175  0.034872   
opp_shots_from_sustained_attacks_pct -0.421411  0.144110 -0.037073   
turnover_line_height_m                0.583990 -0.211519  0.215747   

                                          Factor_4      Factor_5  \
team_id                               1.00

In [6]:
import pandas as pd, json
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

df = pd.read_csv("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/football_test/team_stats.csv")
num = df.select_dtypes(include=["float64","int64"]).dropna(axis=1)
X = StandardScaler().fit_transform(num)

fa = FactorAnalysis(n_components=10, random_state=0).fit(X)
factors = fa.transform(X)

scores = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=0).fit(factors)
    scores.append((k, silhouette_score(factors, km.labels_)))

print(scores)

km = KMeans(n_clusters=7, random_state=0).fit(factors)
labels = km.labels_

centroids = pd.DataFrame(km.cluster_centers_,
                         columns=[f"F{i+1}" for i in range(10)])
print(centroids)

[(2, np.float64(0.0679946557654117)), (3, np.float64(0.06220646767114059)), (4, np.float64(0.06059061349996759)), (5, np.float64(0.07207538141836026)), (6, np.float64(0.07113132036278565)), (7, np.float64(0.07260391822363416))]
         F1        F2        F3        F4        F5        F6        F7  \
0 -0.712293 -0.664393 -0.242064  0.401034 -0.802835  0.299904 -0.444435   
1 -0.030147 -0.527046 -0.794900  0.093339  1.294449  0.341928 -0.043521   
2  0.565304  0.799784  0.239101  0.164304  0.314762  0.895058 -0.865513   
3 -0.470042 -0.271486  0.178506 -0.604091  0.346309 -0.046002  0.040177   
4 -0.149929  0.749195 -0.692192 -0.179793 -0.594824 -0.396272  0.561346   
5 -0.065084  0.048631  1.326623  0.136630  0.293292 -0.153219  0.874575   
6  1.678481 -0.319063  0.111608  0.061563 -0.549564 -0.925751 -0.757440   

         F8        F9       F10  
0 -0.033024 -0.308527 -0.366757  
1  0.292995 -0.630001  0.327439  
2 -0.224132  0.372220  0.219645  
3 -0.566786  0.900693 -0.369632  
4

C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.p

Cluster 0 — The Direct & Low-Control Defenders
Overview:
This cluster represents teams that emphasize direct play with limited possession control and generally low attacking output.
Strengths:
They tend to perform relatively well in defensive transitions and maintain compactness without relying on prolonged buildup structures.
Weaknesses:
Weak attacking factors and low possession metrics indicate struggles in chance creation and sustained pressure against opponents.

🟩 Cluster 1 — The High‑Risk, High‑Transition Opportunists
Overview:
These teams rely heavily on transitional moments, taking risks to generate attacking opportunities rather than controlling the game.
Strengths:
They score strongly on transition‑driven expected goals and quick forward actions that create dangerous chances.
Weaknesses:
Their defensive stability is inconsistent, and they often concede high‑value opportunities when transitions break down.

🟦 Cluster 2 — The High‑Tempo, High‑Intensity Controllers
Overview:
This cluster includes teams with strong possession, fast ball circulation, and high defensive and pressing intensity.
Strengths:
They show strong factors for tempo, pressing, and buildup, allowing them to dominate phases of play and limit opponent possession.
Weaknesses:
Their attacking efficiency is sometimes inconsistent despite good control, and they may struggle to convert dominance into goals.

🟧 Cluster 3 — The Deep‑Block Counterattackers
Overview:
These teams defend deep and compact, preferring to absorb pressure and launch direct counterattacks.
Strengths:
They excel in defensive resilience and transition efficiency, particularly in fast break situations.
Weaknesses:
They struggle in sustained possession, chance creation, and pressing higher up the field.

🟨 Cluster 4 — The Build‑Up Technicians with Defensive Gaps
Overview:
This group consists of technically inclined teams that focus on structured buildup play and possession retention.
Strengths:
They perform well in midfield progression, passing structures, and control‑oriented phases of play.
Weaknesses:
Their defensive intensity and organization are weaker, making them vulnerable to high‑pressure or fast‑transition opponents.

🟪 Cluster 5 — The High‑Creativity Attack‑Focused Units
Overview:
These teams stand out for strong attacking chance creation, especially through structured play and dynamic movement in the final third.
Strengths:
They score high on factors tied to creativity, shot generation, and final-third presence.
Weaknesses:
Their defensive stability and pressing efficiency lag behind their attacking output, leaving them open to counterattacks.

🟫 Cluster 6 — The Elite High‑Output Attackers
Overview:
This cluster represents teams with extremely strong attacking profiles, combining volume, efficiency, and consistent threat.
Strengths:
They score very high on finishing, expected goals, and top-tier attacking factor combinations, reflecting elite production.
Weaknesses:
Their defensive transition and pressing metrics show room for improvement, making them susceptible to conceding when their attacks break down.

Real Madrid show strong attacking metrics, including high shots, goals, xG, and final‑third presence, reflecting an elite and consistently dangerous offensive profile.
They are comparatively average or weaker in defensive transition markers—such as opponent xG allowed, reliance on deeper defensive actions, and vulnerability to conceding high‑value shots—indicating moments of defensive instability under pressure.
Overall, Real Madrid combine overwhelming attacking force with occasional defensive openness, producing a high‑impact but sometimes unbalanced performance profile.

In [7]:
import pandas as pd
import json
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import KMeans

# -----------------------------
# 1. Load data
# -----------------------------
df = df

# identify Real Madrid row
real = df[df["club_name"] == "Real Madrid"].reset_index(drop=True)

# numeric data for factor analysis
num = df.select_dtypes(include=["float64", "int64"]).dropna(axis=1)
X = StandardScaler().fit_transform(num)

# -----------------------------
# 2. Factor analysis (10 factors)
# -----------------------------
fa = FactorAnalysis(n_components=10, random_state=0).fit(X)
factors = fa.transform(X)

# Real Madrid factor scores
real_factors = fa.transform(StandardScaler().fit_transform(num))[df.index[df["club_name"]=="Real Madrid"]][0]

# -----------------------------
# 3. Clustering (k = 7)
# -----------------------------
km = KMeans(n_clusters=7, random_state=0).fit(factors)
labels = km.labels_

real_cluster = labels[df.index[df["club_name"]=="Real Madrid"]][0]

# -----------------------------
# 4. Interpretation rules
# (based on centroid directions observed earlier)
# -----------------------------

cluster_descriptions = {
    6: {
        "strength": "Real Madrid show strong attacking metrics, including high shots, goals, xG, and final‑third presence, reflecting an elite and consistently dangerous offensive profile.",
        "weakness": "They are comparatively average or weaker in defensive transition markers such as opponent xG allowed and vulnerability to conceding high‑value shots under pressure.",
        "summary": "Overall, Real Madrid combine overwhelming attacking force with occasional defensive openness, producing a high‑impact but sometimes unbalanced performance profile."
    }
    # You can add entries for other clusters if needed.
}

# -----------------------------
# 5. Select the correct description
# -----------------------------
desc = cluster_descriptions.get(real_cluster, {
    "strength": "Strength profile unavailable.",
    "weakness": "Weakness profile unavailable.",
    "summary": "No cluster summary available."
})

# -----------------------------
# 6. Print final 3‑sentence summary
# -----------------------------
print(desc["strength"])
print(desc["weakness"])
print(desc["summary"])

Strength profile unavailable.
Weakness profile unavailable.
No cluster summary available.


C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
